# Supplementary Table 4

Enterotype Dysbiosis Score (EDS) displays in the PREDICT cohort: the earlier 20-feature, 259-observation selection with its 85-patient display, and the three post hoc metabolite-enzyme pairs of Supplementary Fig. 10a with their associations, saved full-family q values and 192 descriptive adjustment estimates. This notebook rebuilds all eleven sheets from `source_data/`, compares them cell for cell with the published workbook (clinical columns withheld), and recomputes the statistics the table reports.

Run from the repository root: `python paper.py build STab4`.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
d = panels.load(REPO)
ctrl = panels.load_controlled(d)

## Table

Eleven sheets. All20_features_259_observations to 85_patient_z: the earlier selection of 20 features (host proteins, microbial KEGG pathways, metabolites) across 259 matched observations (repeated visits included) and its display in 85 patients, one observation per patient (Fig. 4h, Supplementary Fig. 10b). Focused_raw to Focused_adjustments: the three metabolite-enzyme pairs of Supplementary Fig. 10a. Supports the legend (v28d line 481) and "examined three metabolite–enzyme pairs selected post hoc from an exploratory screen of this cohort (Supplementary Fig. 10a, Supplementary Table 4)" (v28d line 137). Unit: patient (85) or observation (259). No test in this cell.

In the repository version, acquisition and microbiome sample identifiers, clinical group and antibiotic state are withheld; sample identifiers are replaced by observation index (1-259) or patient display position (1-85, EDS order).

In [2]:
sheets = panels.build_sheets(d)
path = panels.write_workbook(sheets, OUT / "Supplementary_Table_4.xlsx")
for name, rows in sheets.items():
    print(f"{name:34s} {len(rows):4d} rows x {max(len(r) for r in rows):4d} columns")

Read_me                               9 rows x    1 columns
All20_features_259_observations      21 rows x  260 columns
Fixed_feature_key                    21 rows x    5 columns
Observation_metadata                260 rows x    5 columns
85_patient_order                     86 rows x    6 columns
85_patient_raw                       21 rows x   86 columns
85_patient_z                         21 rows x   86 columns
Focused_raw                          86 rows x   12 columns
Focused_features                      7 rows x    6 columns
Focused_pairs                         4 rows x   11 columns
Focused_adjustments                 193 rows x    8 columns


## Statistics: the three pairs

Spearman correlation of each metabolite with the protein signal of its enzyme (KO), over patients with positive finite values for both (n = 85, 84, 85 patients; two-sided t test, df = n - 2). "Spearman ρ=−0.53, −0.44 and −0.35, with 84 patients for tryptophan" (v28d line 137). q values are recomputed by Benjamini-Hochberg over the complete native KO linked-pair family (1,042 tested pairs) from the saved screen P values; the joint-adjusted model is the same family under adjustment for human-assigned share, protein-group depth, antibiotic status and clinical group (v28d line 137: "none passed multiple-testing correction after joint adjustment").

In [3]:
una = panels.stat_focused_unadjusted(d)
print(una.to_string(index=False))
fq, qdiff = panels.stat_family_q(d)
print(fq.to_string(index=False))
print("largest |recomputed q - saved q_augmented| over the whole screen:", qdiff)
sheet_pairs = d["selected_pair_summary"]
assert np.allclose(una.rho, sheet_pairs.rho, rtol=0, atol=1e-12) and (una.n.to_numpy() == sheet_pairs.n.to_numpy()).all()
assert np.allclose(fq[fq.model == "unadjusted"].q, sheet_pairs.full_family_q, rtol=0, atol=1e-12)
assert np.allclose(fq[fq.model != "unadjusted"].q, sheet_pairs.joint_adjusted_full_family_q, rtol=0, atol=1e-12)
assert np.allclose(fq[fq.model != "unadjusted"].rho, sheet_pairs.joint_adjusted_rho, rtol=0, atol=1e-12)
joint = panels.stat_joint_adjusted(d, ctrl)
if joint is not None:
    print(joint.to_string(index=False))
    assert np.allclose(joint.rho, sheet_pairs.joint_adjusted_rho, rtol=0, atol=1e-12)

   metabolite     KO                                          label  n       rho         t  df            P  scipy_rho
HILIC_FT23003 K01812          Glucuronic acid-Glucuronate isomerase 85 -0.528669 -5.674183  83 1.984389e-07  -0.528669
HILIC_FT24070 K01667                       Tryptophan-Tryptophanase 84 -0.439627 -4.432290  82 2.864133e-05  -0.439627
    C18_F2423 K01442 Glycochenodeoxycholic acid-Bile salt hydrolase 85 -0.347313 -3.374218  83 1.127294e-03  -0.347313
   metabolite     KO                          model  family_size  n       rho            P        q
HILIC_FT23003 K01812                     unadjusted         1042 85 -0.528669 1.984389e-07 0.000122
HILIC_FT23003 K01812 host_depth_antibiotic_clinical         1042 85 -0.249888 2.634974e-02 0.496496
HILIC_FT24070 K01667                     unadjusted         1042 84 -0.439627 2.864133e-05 0.001571
HILIC_FT24070 K01667 host_depth_antibiotic_clinical         1042 84 -0.086069 4.536998e-01 0.866927
    C18_F2423 K01442    

## Statistics: the six displayed features

Spearman correlation of each displayed feature with EDS over patients with a positive value (n = 85 or 84 patients), and Benjamini-Hochberg q recomputed in the complete native unadjusted EDS family of its layer (283 metabolites, 6,965 KOs). Sheet Focused_features (v28d line 481: "the feature associations").

In [4]:
ff = panels.stat_focused_features(d)
print(ff.to_string(index=False))
assert np.allclose(ff.rho, d["focused_features"].EDS_rho, rtol=0, atol=1e-12)
assert np.allclose(ff.q, d["focused_features"].EDS_full_family_q, rtol=0, atol=1e-12)
assert (ff.n.to_numpy() == d["focused_features"].n_positive.to_numpy()).all()

      feature      layer  n       rho         t  df        P        q  family_size  saved_q
HILIC_FT23003 Metabolite 85  0.422267  4.243972  83 0.000057 0.001238          283 0.001238
       K01812         KO 85 -0.453145 -4.631122  83 0.000013 0.001819         6965 0.001819
HILIC_FT24070 Metabolite 85  0.392894  3.892453  83 0.000200 0.002757          283 0.002757
       K01667         KO 84 -0.296250 -2.808739  82 0.006215 0.038107         6965 0.038107
    C18_F2423 Metabolite 85  0.337333  3.264605  83 0.001594 0.009803          283 0.009803
       K01442         KO 85 -0.293029 -2.792188  83 0.006496 0.038940         6965 0.038940


## Statistics: 192 adjustment estimates

Partial rank correlations for all 16 combinations of human-assigned share, log2 protein-group depth, antibiotic status and clinical group, under native and annotated-relative normalization, on the pair-specific complete subsets (85, 84, 85 patients) and the common 84-patient subset: 3 x 2 x 2 x 16 = 192 point estimates, without P values or intervals (v28d line 799). "Adjustment for protein-group depth alone reduced them to ρ=−0.24, −0.06 and −0.10" (v28d line 137). Without the controlled input only the 48 estimates that use neither antibiotic status nor clinical group are recomputed.

In [5]:
grid, n_common = panels.stat_adjustment_grid(d, ctrl)
ref = d["separate_adjustments_all_models"]
assert (grid[["metabolite", "KO", "scope", "normalization", "model", "n"]].to_numpy() == ref[["metabolite", "KO", "scope", "normalization", "model", "n"]].to_numpy()).all()
ok = np.isfinite(grid.rho.to_numpy())
print("estimates recomputed:", int(ok.sum()), "of", len(grid), "; common subset:", n_common)
print("largest |recomputed - sheet| rho:", float(np.max(np.abs(grid.rho.to_numpy()[ok] - ref.rho.to_numpy()[ok]))))
depth = grid[(grid.scope == "pair_complete") & (grid.normalization == "native") & (grid.model.isin(["none", "depth"]))]
print(depth.to_string(index=False))

estimates recomputed: 192 of 192 ; common subset: 84
largest |recomputed - sheet| rho: 0.0
   metabolite     KO         scope normalization model  n       rho  delta_from_unadjusted
HILIC_FT23003 K01812 pair_complete        native  none 85 -0.528669               0.000000
HILIC_FT23003 K01812 pair_complete        native depth 85 -0.237821               0.290849
HILIC_FT24070 K01667 pair_complete        native  none 84 -0.439627               0.000000
HILIC_FT24070 K01667 pair_complete        native depth 84 -0.060830               0.378797
    C18_F2423 K01442 pair_complete        native  none 85 -0.347313               0.000000
    C18_F2423 K01442 pair_complete        native depth 85 -0.095285               0.252027


## Statistics: overview features and z scores

Fixed_feature_key: positive patients and Spearman rho with EDS over the 85 patients (19 of 20 features have a positive value; HV366 has none). 85_patient_z and Focused_raw display values: within-feature z over positive finite values, population SD (v28d line 801).

In [6]:
ov = panels.stat_overview_features(d)
print(ov.to_string(index=False))
assert (ov.n_positive == ov.n_sheet).all() and np.nanmax(ov.abs_diff) < 1e-12
print(panels.stat_z(d))

               feature_id  n_positive  n_sheet       rho  rho_sheet  abs_diff
    sp|O14558|HSPB6_HUMAN          29       29 -0.544828  -0.544828       0.0
     sp|Q13642|FHL1_HUMAN          29       29 -0.172906  -0.172906       0.0
     sp|O60635|TSN1_HUMAN          38       38 -0.310117  -0.310117       0.0
sp|A0A0C4DH42|HV366_HUMAN           0        0       NaN        NaN       NaN
     sp|P15144|AMPN_HUMAN          83       83  0.276380   0.276380       0.0
     sp|P04179|SODM_HUMAN          43       43  0.455544   0.455544       0.0
                 map00910          85       85 -0.362251  -0.362251       0.0
                 map00220          85       85 -0.411675  -0.411675       0.0
                 map01210          85       85 -0.366159  -0.366159       0.0
                 map01212          85       85 -0.399578  -0.399578       0.0
                 map00720          85       85 -0.375227  -0.375227       0.0
                 map00250          85       85 -0.332213  -0.332

## Statistics: five-pair decomposition (Source Data only)

v28d line 799 describes an earlier decomposition of five metabolite-KO or metabolite-pathway pairs (all 16 covariate combinations, a common 84-patient subset, 1,000 patient bootstrap resamples). It is not part of this workbook; its saved model table is read here to trace the counts. The bootstrap is not rerun (see OPEN_ITEMS.md).

In [7]:
five = panels.stat_five_pair(d)
print(five)

{'pairs': 5, 'combinations': 16, 'common_n': 84, 'draws': 1000, 'rows': 320, 'scopes': ['all_pairs_complete', 'pair_complete']}


## Checks and numbers

Cell-for-cell comparison of the rebuilt workbook with the published workbook (redacted copy in `expected/`). With `FASTALAKE_CONTROLLED_DIR`, the complete published layout is rebuilt and compared with the published workbook itself. Then every number of `NUMBERS.tsv`, in row order.

In [8]:
(UNIT / "checks").mkdir(exist_ok=True)
cmp = panels.compare_workbooks(OUT / "Supplementary_Table_4.xlsx", UNIT / "expected" / "Supplementary_Table_4_redacted.xlsx")
print(cmp.to_string(index=False))
cmp.to_csv(UNIT / "checks" / "workbook_vs_published_redacted.tsv", sep="\t", index=False)
assert cmp.differing.sum() == 0 and (cmp.rows == cmp.reference_rows).all()
if ctrl is not None:
    full = panels.write_workbook(panels.build_sheets(d, ctrl), OUT / "Supplementary_Table_4_full.xlsx")
    cmpf = panels.compare_workbooks(full, ctrl["published"])
    print(cmpf.to_string(index=False))
    cmpf.to_csv(UNIT / "checks" / "workbook_vs_published_full.tsv", sep="\t", index=False)
    assert cmpf.differing.sum() == 0 and (cmpf.rows == cmpf.reference_rows).all()
else:
    print("full-layout check SKIPPED: controlled input absent (committed receipt kept)")

cnt = panels.stat_counts(d, sheets)
dep = depth[depth.model == "depth"].set_index("metabolite").rho
grid_models = grid.groupby(["metabolite", "scope", "normalization"]).model.nunique().unique()
assert len(grid_models) == 1
pair_n = una.set_index("metabolite").n
nums = [
    panels.word(cnt["focused_pairs"]),                       # 137 three
    cnt["patients"],                                          # 137 85
    *una.rho.tolist(),                                        # 137 rho x3
    int(pair_n["HILIC_FT24070"]),                             # 137 84
    *[float(dep[m]) for m, *_ in panels.PAIRS],                      # 137 depth-only rho x3
    cnt["original_features"], cnt["original_observations"], cnt["patients"],   # 481
    panels.word(cnt["focused_pairs"]), panels.word(cnt["focused_measurements"]), cnt["adjustment_estimates"],
    panels.word(five["pairs"]), five["combinations"], five["common_n"], five["draws"],   # 799 five-pair
    panels.word(cnt["focused_pairs"]), int(grid_models[0]),   # 799 three pairs, 16 combinations
    *[int(pair_n[m]) for m, *_ in panels.PAIRS], n_common, len(grid),   # 799 85, 84, 85, 84, 192
]
for i, v in enumerate(nums, 1):
    print(f"{i}\t{v}")

                          sheet  rows  reference_rows  columns  cells  differing first_difference
                        Read_me     9               9        1      9          0                 
All20_features_259_observations    21              21      260   5460          0                 
              Fixed_feature_key    21              21        5    105          0                 
           Observation_metadata   260             260        5   1300          0                 
               85_patient_order    86              86        6    516          0                 
                 85_patient_raw    21              21       86   1806          0                 
                   85_patient_z    21              21       86   1806          0                 
                    Focused_raw    86              86       12   1032          0                 
               Focused_features     7               7        6     42          0                 
                  Fo

                          sheet  rows  reference_rows  columns  cells  differing first_difference
                        Read_me     9               9        1      9          0                 
All20_features_259_observations    21              21      260   5460          0                 
              Fixed_feature_key    21              21        5    105          0                 
           Observation_metadata   260             260        8   2080          0                 
               85_patient_order    86              86        9    774          0                 
                 85_patient_raw    21              21       86   1806          0                 
                   85_patient_z    21              21       86   1806          0                 
                    Focused_raw    86              86       15   1290          0                 
               Focused_features     7               7        6     42          0                 
                  Fo